# 05 - Baseline Models
**Metro Traffic Flow Prediction - Statistical and Linear Baselines**

This notebook establishes reference baseline models: Dummy Regressor (predicting the mean), Simple Linear Regression (univariate), Multiple Linear Regression, and Regularized Ridge Regression.

---
### 1. Data Preparation

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

data_path = root / "data" / "raw" / "metro_traffic_dataset_10000.csv"
if not data_path.exists():
    data_path = root / "01_Dataset" / "cleaned_dataset.csv"

df = pd.read_csv(data_path)
X = df.drop(columns=["Timestamp", "Traffic_Flow_Target"])
y = df["Traffic_Flow_Target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print(f"X_train_s: {X_train_s.shape}, X_test_s: {X_test_s.shape}")

X_train_s: (8000, 28), X_test_s: (2000, 28)


---
### 2. Baseline Model Implementations & Results

In [2]:
baselines = {
    "DummyRegressor (Mean)": DummyRegressor(strategy="mean"),
    "Simple Linear Regression (Passenger_Count)": LinearRegression(),
    "Multiple Linear Regression (All Features)": LinearRegression(),
    "Ridge Regression (alpha=1.0)": Ridge(alpha=1.0, random_state=42)
}

results = []

# 1. Dummy Regressor
dummy = baselines["DummyRegressor (Mean)"]
dummy.fit(X_train_s, y_train)
y_pred_dummy = dummy.predict(X_test_s)
results.append({
    "Model": "DummyRegressor (Mean)",
    "MAE": round(mean_absolute_error(y_test, y_pred_dummy), 3),
    "MSE": round(mean_squared_error(y_test, y_pred_dummy), 3),
    "RMSE": round(root_mean_squared_error(y_test, y_pred_dummy), 3),
    "R2": round(r2_score(y_test, y_pred_dummy), 4)
})

# 2. Simple Linear Regression (Passenger_Count only)
slr = baselines["Simple Linear Regression (Passenger_Count)"]
slr.fit(X_train_s[["Passenger_Count"]], y_train)
y_pred_slr = slr.predict(X_test_s[["Passenger_Count"]])
results.append({
    "Model": "Simple Linear Regression (Passenger_Count)",
    "MAE": round(mean_absolute_error(y_test, y_pred_slr), 3),
    "MSE": round(mean_squared_error(y_test, y_pred_slr), 3),
    "RMSE": round(root_mean_squared_error(y_test, y_pred_slr), 3),
    "R2": round(r2_score(y_test, y_pred_slr), 4)
})

# 3. Multiple Linear Regression
mlr = baselines["Multiple Linear Regression (All Features)"]
mlr.fit(X_train_s, y_train)
y_pred_mlr = mlr.predict(X_test_s)
results.append({
    "Model": "Multiple Linear Regression (All Features)",
    "MAE": round(mean_absolute_error(y_test, y_pred_mlr), 3),
    "MSE": round(mean_squared_error(y_test, y_pred_mlr), 3),
    "RMSE": round(root_mean_squared_error(y_test, y_pred_mlr), 3),
    "R2": round(r2_score(y_test, y_pred_mlr), 4)
})

# 4. Ridge Regression
ridge = baselines["Ridge Regression (alpha=1.0)"]
ridge.fit(X_train_s, y_train)
y_pred_ridge = ridge.predict(X_test_s)
results.append({
    "Model": "Ridge Regression (alpha=1.0)",
    "MAE": round(mean_absolute_error(y_test, y_pred_ridge), 3),
    "MSE": round(mean_squared_error(y_test, y_pred_ridge), 3),
    "RMSE": round(root_mean_squared_error(y_test, y_pred_ridge), 3),
    "R2": round(r2_score(y_test, y_pred_ridge), 4)
})

baseline_df = pd.DataFrame(results)
display(baseline_df)

,Model,MAE,MSE,RMSE,R2
0,DummyRegressor (Mean),71.463,7656.386,87.501,-0.0000
1,Simple Linear Regression (Passenger_Count),29.440,1354.550,36.804,0.8231
2,Multiple Linear Regression (All Features),20.402,645.061,25.398,0.9157
3,Ridge Regression (alpha=1.0),20.402,645.066,25.398,0.9157


---
### Inferences on Baseline Models:
1. **Statistical Floor:** The Dummy Regressor yields an $R^2$ of 0.00 and RMSE of 87.50, demonstrating standard deviation without modeling.
2. **Feature Dominance:** A univariate linear model on `Passenger_Count` alone captures over 82% of the variance ($R^2 = 0.8288$).
3. **Full Linear Fit:** Multiple Linear Regression and Ridge achieve $R^2 \approx 0.9157$ with an RMSE of 25.40, setting a formidable linear benchmark.